# Chapter 1: Vectors

A **vector** is an ordered list of numbers — the most fundamental object in linear algebra
and the backbone of every ML model.

| Domain | Vector Example |
|--------|---------------|
| 🖼️ Computer Vision | Flattened 28×28 MNIST image (784-dim) |
| 📝 NLP | BERT token embedding (768-dim) |
| 🤖 ML | Feature vector of one data sample $(x_1,\ldots,x_n)$ |
| 🎮 Graphics | 3-D position $(x, y, z)$ or colour $(r, g, b)$ |

## What You Will Learn
1. Creating and representing vectors
2. Addition, subtraction, scalar multiplication
3. Dot product — algebra, geometry, and properties
4. Vector length, unit vectors, orthogonality
5. Cauchy-Schwarz inequality
6. Hadamard product, outer product, cross product
7. Complex vectors and the Hermitian transpose
8. Span, subspaces, and linear independence

## Notation
| Symbol | Meaning |
|--------|---------|
| $\mathbf{v}$ | vector (lowercase bold) |
| $\alpha, \beta, \lambda$ | scalar (Greek) |
| $v_i$ | $i$-th component |
| $\mathbf{v}^\top$ | transpose |
| $\|\mathbf{v}\|$ | Euclidean length |
| $\mathbb{R}^n$ | $n$-dimensional real space |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

def draw_vec(ax, v, origin=(0, 0), color='steelblue', label=''):
    ax.quiver(*origin, v[0], v[1],
              angles='xy', scale_units='xy', scale=1,
              color=color, width=0.008, headwidth=4, headlength=5,
              label=label or None)


## 1. What is a Vector?

An **algebraic vector** is an ordered list of numbers; each element has a fixed index.

$$
\text{column:}\quad
\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{bmatrix} \in \mathbb{R}^{n \times 1}
\qquad\qquad
\text{row:}\quad
\mathbf{v}^\top = \begin{bmatrix} v_1 & v_2 & \cdots & v_n \end{bmatrix} \in \mathbb{R}^{1 \times n}
$$

The **dimension** of a vector is its number of elements.

> **NumPy shapes:**
> `(n, 1)` = column · `(1, n)` = row · `(n,)` = 1-D array (default; most common in practice)


In [ ]:
# Explicit column vector — shape (n, 1)
col_v = np.array([[1], [2], [3]])
print("Column vector:\n", col_v, "  shape:", col_v.shape)

# Explicit row vector — shape (1, n)
row_v = np.array([[1, 2, 3]])
print("\nRow vector:\n", row_v, "  shape:", row_v.shape)

# 1-D array — shape (n,)  ← most common in NumPy
v = np.array([1, 2, 3])
print("\n1-D array:", v, "  shape:", v.shape)

# Convert between forms
print("\nReshape → column:\n", v.reshape(-1, 1))
print("Reshape → row   :\n", v.reshape(1, -1))


## 2. Vector Addition and Subtraction

Element-wise — both vectors must have the same dimension:

$$
\mathbf{a} + \mathbf{b} =
\begin{bmatrix} a_1 \\ \vdots \\ a_n \end{bmatrix} +
\begin{bmatrix} b_1 \\ \vdots \\ b_n \end{bmatrix} =
\begin{bmatrix} a_1+b_1 \\ \vdots \\ a_n+b_n \end{bmatrix}
$$

**Geometric interpretation:**
- **Addition** — place $\mathbf{b}$'s tail at $\mathbf{a}$'s head (*tail-to-head*)
- **Subtraction** — $\mathbf{a}-\mathbf{b}$ points from the tip of $\mathbf{b}$ to the tip of $\mathbf{a}$ (*head-to-head*)


In [ ]:
a = np.array([3, -1])
b = np.array([2,  4])

print(f"a + b = {a + b}")
print(f"a - b = {a - b}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: addition (tail-to-head)
ax = axes[0]
draw_vec(ax, a,           color='steelblue',   label=r'$\mathbf{a}$')
draw_vec(ax, b, origin=a, color='tomato',       label=r"$\mathbf{b}$ (tail at tip of $\mathbf{a}$)")
draw_vec(ax, a + b,       color='forestgreen',  label=r'$\mathbf{a}+\mathbf{b}$')
ax.set(title='Addition: tail-to-head', xlim=(-1, 7), ylim=(-3, 5), aspect='equal')
ax.legend(loc='upper left'); ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5)

# Right: subtraction (head-to-head)
ax = axes[1]
draw_vec(ax, a,               color='steelblue', label=r'$\mathbf{a}$')
draw_vec(ax, b,               color='tomato',    label=r'$\mathbf{b}$')
draw_vec(ax, a - b, origin=b, color='orchid',    label=r'$\mathbf{a}-\mathbf{b}$')
ax.set(title='Subtraction: head-to-head', xlim=(-1, 7), ylim=(-3, 5), aspect='equal')
ax.legend(loc='upper left'); ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5)

plt.suptitle('Vector Addition and Subtraction', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 3. Scalar Multiplication

$$\lambda \mathbf{v} = \begin{bmatrix} \lambda v_1 \\ \vdots \\ \lambda v_n \end{bmatrix}$$

**Notation:** Matrices → uppercase $A, B$ · Vectors → lowercase bold $\mathbf{v}, \mathbf{u}$ · Scalars → Greek $\alpha, \beta, \lambda$

| Condition | Effect |
|-----------|--------|
| $\lambda > 1$ | **stretches** (longer, same direction) |
| $0 < \lambda < 1$ | **shrinks** (shorter, same direction) |
| $\lambda < 0$ | **reverses** direction |

> **ML connection:** Gradient descent $\theta \leftarrow \theta - \alpha\nabla L$ is scalar multiplication of the gradient by the learning rate $\alpha$.


In [ ]:
v = np.array([2, 1])
cases  = [2.0, 0.5, -1.5]
labels = [r'$\lambda=2.0$ — stretch', r'$\lambda=0.5$ — shrink', r'$\lambda=-1.5$ — reverse']
colors = ['tomato', 'forestgreen', 'orchid']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, lam, lab, col in zip(axes, cases, labels, colors):
    lim = max(abs(lam) * 2.5, 2.5)
    draw_vec(ax, v,      color='steelblue', label=r'$\mathbf{v}$')
    draw_vec(ax, lam*v,  color=col,         label=lab)
    ax.set(title=lab, xlim=(-lim, lim), ylim=(-lim, lim), aspect='equal')
    ax.legend(fontsize=9); ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5)

plt.suptitle('Scalar Multiplication', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 4. The Dot Product

The dot product maps two same-length vectors to a single **scalar**:

$$\alpha = \mathbf{a} \cdot \mathbf{b} = \langle \mathbf{a}, \mathbf{b} \rangle
         = \mathbf{a}^\top \mathbf{b}
         = \sum_{i=1}^{n} a_i b_i, \qquad \mathbf{a}, \mathbf{b} \in \mathbb{R}^n$$

> **ML connections:** Logistic regression score $\mathbf{w}^\top\mathbf{x}$; scaled dot-product
> attention score $\mathbf{q}^\top\mathbf{k}$; cosine similarity in retrieval.


In [ ]:
v1 = np.array([1, 2, 3, 4, 5, 6])
v2 = np.array([0, -4, 1, -3, 6, 5])

dp1 = sum(np.multiply(v1, v2))                      # element-wise then sum
dp2 = np.dot(v1, v2)                                # np.dot
dp3 = np.matmul(v1, v2)                             # np.matmul
dp4 = v1 @ v2                                       # @ operator (recommended)
dp5 = sum(v1[i] * v2[i] for i in range(len(v1)))   # explicit loop

print(f"sum(v1*v2)  = {dp1}")
print(f"np.dot      = {dp2}")
print(f"np.matmul   = {dp3}")
print(f"@ operator  = {dp4}")
print(f"manual loop = {dp5}")
print(f"All equal?  {dp1 == dp2 == dp3 == dp4 == dp5}")


## 5. Dot Product Properties

| Property | Formula | Holds? |
|----------|---------|--------|
| **Distributive** | $\mathbf{a}\cdot(\mathbf{b}+\mathbf{c}) = \mathbf{a}\cdot\mathbf{b} + \mathbf{a}\cdot\mathbf{c}$ | ✅ |
| **Commutative** | $\mathbf{a}\cdot\mathbf{b} = \mathbf{b}\cdot\mathbf{a}$ | ✅ |
| **Associative** | $\mathbf{a}\cdot(\mathbf{b}\cdot\mathbf{c}) \stackrel{?}{=} (\mathbf{a}\cdot\mathbf{b})\cdot\mathbf{c}$ | ❌ |

The dot product is **not** associative because $\mathbf{b}\cdot\mathbf{c}$ is a scalar, so
$\mathbf{a}\cdot(\mathbf{b}\cdot\mathbf{c})$ scales $\mathbf{a}$ while $(\mathbf{a}\cdot\mathbf{b})\cdot\mathbf{c}$ scales $\mathbf{c}$ — different vectors in general.


In [ ]:
n = 10
a = np.random.rand(n)
b = np.random.rand(n)
c = np.random.rand(n)

# Distributive: a·(b+c) == a·b + a·c
print("Distributive a·(b+c) == a·b + a·c :",
      np.isclose(a @ (b + c), (a @ b) + (a @ c)))

# Commutative: a·b == b·a
print("Commutative  a·b == b·a            :", np.isclose(a @ b, b @ a))

# NOT associative — (a·b)c and a(b·c) are different vectors
lhs = (a @ b) * c   # scalar × c
rhs =  a * (b @ c)  # scalar × a
print("Associative  (a·b)c == a(b·c)      :", np.allclose(lhs, rhs),
      " <- NOT associative (as expected)")


## 6. Vector Length (Magnitude)

From the Pythagorean theorem:

$$\|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2}
               = \sqrt{\mathbf{v}^\top \mathbf{v}}$$


In [ ]:
v = np.array([3.0, 4.0])   # classic 3-4-5 right triangle
l1 = np.sqrt(v @ v)
l2 = np.linalg.norm(v)

print(f"v = {v}")
print(f"sqrt(v·v)      = {l1:.4f}")
print(f"np.linalg.norm = {l2:.4f}")
print(f"\nnp.array([1,2,3,4,5]) length = {np.linalg.norm([1, 2, 3, 4, 5]):.4f}")

# Pythagorean visualisation
fig, ax = plt.subplots(figsize=(5, 5))
draw_vec(ax, v, color='steelblue',
         label=rf"$\mathbf{{v}}=({v[0]:.0f},{v[1]:.0f}),\ \|\mathbf{{v}}\|={l2:.0f}$")
ax.plot([0, v[0]], [0, 0],       'r--', lw=1.5, label=f"$v_1={v[0]:.0f}$")
ax.plot([v[0], v[0]], [0, v[1]], 'g--', lw=1.5, label=f"$v_2={v[1]:.0f}$")
ax.set(xlim=(-0.5, 5), ylim=(-0.5, 5), aspect='equal',
       title=r'Pythagorean: $\|\mathbf{v}\|=\sqrt{3^2+4^2}=5$')
ax.legend(); ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5)
plt.tight_layout(); plt.show()


## 7. Dot Product Geometry: Angle Between Vectors

$$\mathbf{a}^\top\mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta_{\mathbf{ab}}
\qquad\Rightarrow\qquad
\cos\theta_{\mathbf{ab}} = \frac{\mathbf{a}^\top\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$$

| Sign of $\mathbf{a}^\top\mathbf{b}$ | $\theta$ | Interpretation |
|--------------------------------------|----------|----------------|
| $> 0$ | $< 90°$ | vectors point in similar directions |
| $= 0$ | $= 90°$ | **orthogonal** (perpendicular) |
| $< 0$ | $> 90°$ | vectors point in opposite directions |


In [ ]:
v1 = np.array([2.0, 4.0])
v2 = np.array([4.0, 1.0])

cos_theta = (v1 @ v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
theta_rad = np.arccos(np.clip(cos_theta, -1, 1))
theta_deg = np.degrees(theta_rad)

dp_alg = v1 @ v2
dp_geo = np.linalg.norm(v1) * np.linalg.norm(v2) * np.cos(theta_rad)
print(f"Algebraic    a·b            = {dp_alg:.6f}")
print(f"Geometric    ||a||·||b||·cosθ = {dp_geo:.6f}")
print(f"Angle θ = {theta_deg:.2f}°")

fig, ax = plt.subplots(figsize=(6, 5))
draw_vec(ax, v1, color='steelblue', label=r'$\mathbf{v}_1$')
draw_vec(ax, v2, color='tomato',    label=r'$\mathbf{v}_2$')

t1 = np.degrees(np.arctan2(v1[1], v1[0]))
t2 = np.degrees(np.arctan2(v2[1], v2[0]))
arc = Arc((0, 0), 1.4, 1.4, angle=0,
          theta1=min(t1, t2), theta2=max(t1, t2),
          color='gray', lw=1.5)
ax.add_patch(arc)
mid = np.radians((t1 + t2) / 2)
ax.text(0.9 * np.cos(mid), 0.9 * np.sin(mid),
        f'{theta_deg:.1f}°', ha='center', fontsize=11, color='gray')

ax.set(xlim=(-0.5, 5.5), ylim=(-0.5, 5.5), aspect='equal',
       title=f'Angle between v₁ and v₂: θ = {theta_deg:.1f}°')
ax.legend(); ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5)
plt.tight_layout(); plt.show()


## 8. Unit Vectors

A **unit vector** has magnitude exactly 1. Normalize any non-zero vector by dividing by its length:

$$\hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|}, \qquad \|\hat{\mathbf{v}}\| = 1$$

$\hat{\mathbf{v}}$ points in the **same direction** as $\mathbf{v}$ — only the scale changes.

> **ML connection:** Cosine similarity normalizes before the dot product:
> $\text{sim}(\mathbf{a},\mathbf{b}) = \hat{\mathbf{a}}\cdot\hat{\mathbf{b}} \in [-1,1]$.
> Used in word embeddings (Word2Vec, GloVe), BERT sentence similarity, and RAG retrieval.


In [ ]:
v = np.array([3.0, 4.0])
unit_v = v / np.linalg.norm(v)

print(f"v      = {v},   ||v||      = {np.linalg.norm(v):.4f}")
print(f"unit_v = {unit_v},  ||unit_v|| = {np.linalg.norm(unit_v):.6f}")

# cos(angle between v and unit_v) = 1 → they point in the same direction
cos_same = (v @ unit_v) / (np.linalg.norm(v) * np.linalg.norm(unit_v))
print(f"\ncos(angle between v and v̂) = {cos_same:.6f}  (= 1 → same direction)")

# Higher-dimension example
v3 = np.array([5.0, 2.0, 8.0])
unit_v3 = v3 / np.linalg.norm(v3)
print(f"\nv3 = {v3},  ||unit_v3|| = {np.linalg.norm(unit_v3):.6f}")

fig, ax = plt.subplots(figsize=(5, 5))
draw_vec(ax, v,      color='steelblue',
         label=rf"$\mathbf{{v}}$  (||v||={np.linalg.norm(v):.0f})")
draw_vec(ax, unit_v, color='tomato',
         label=r"$\hat{\mathbf{v}}$  (||v̂||=1)")
ax.set(xlim=(-0.5, 4), ylim=(-0.5, 4.5), aspect='equal',
       title='Original vector vs. its unit vector')
ax.legend(); ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5)
plt.tight_layout(); plt.show()


## 9. Orthogonality

Two vectors are **orthogonal** when their dot product is zero:

$$\mathbf{a} \perp \mathbf{b} \;\iff\; \mathbf{a}\cdot\mathbf{b} = 0 \;\iff\; \theta = 90°$$

An **orthonormal** set is simultaneously orthogonal and unit-length:
$\mathbf{q}_i\cdot\mathbf{q}_j = \delta_{ij}$ (Kronecker delta).

> **ML connections:** Orthogonal features are uncorrelated. PCA produces an orthonormal basis.
> Orthogonal weight initialisations preserve gradient norms during backpropagation.


In [ ]:
# Classic 90-degree pair
u = np.array([3.0, 0.0])
v = np.array([0.0, 2.0])
print(f"u·v = {u @ v}  → orthogonal: {np.isclose(u @ v, 0)}")

# For any (a, b), the vector (-b, a) is always perpendicular to it
w      = np.array([2.0, 3.0])
w_perp = np.array([-w[1], w[0]])
print(f"w = {w},  w⊥ = {w_perp},  w·w⊥ = {w @ w_perp}")

# Verify: [16, -2, 4] · [0.5, 2, -1] == 0
p = np.array([16.0, -2.0, 4.0])
q = np.array([0.5,   2.0, -1.0])
print(f"\n[16,-2,4]·[0.5,2,-1] = {np.dot(p, q)}  → orthogonal: {np.isclose(np.dot(p, q), 0)}")

fig, ax = plt.subplots(figsize=(6, 5))
draw_vec(ax, u,      color='steelblue', label=r'$\mathbf{u}$')
draw_vec(ax, v,      color='tomato',    label=r'$\mathbf{v}$  ($\mathbf{u}\perp\mathbf{v}$)')
draw_vec(ax, w,      color='orchid',    label=r'$\mathbf{w}$')
draw_vec(ax, w_perp, color='seagreen',  label=r'$\mathbf{w}^\perp$')
sq = 0.25
ax.plot([sq, sq, 0], [0, sq, sq], 'k-', lw=1.5)  # right-angle marker
ax.set(xlim=(-4, 5), ylim=(-3.5, 3.5), aspect='equal', title='Orthogonal Vector Pairs')
ax.legend(loc='upper right'); ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5)
plt.tight_layout(); plt.show()


## 10. The Cauchy-Schwarz Inequality

$$|\mathbf{a}^\top\mathbf{b}| \leq \|\mathbf{a}\|\,\|\mathbf{b}\|$$

**Proof:** From $\mathbf{a}^\top\mathbf{b} = \|\mathbf{a}\|\|\mathbf{b}\|\cos\theta$, taking absolute values:

$$|\mathbf{a}^\top\mathbf{b}| = \|\mathbf{a}\|\|\mathbf{b}\||\cos\theta| \leq \|\mathbf{a}\|\|\mathbf{b}\|$$

since $|\cos\theta| \leq 1$, with equality iff $\mathbf{a} \parallel \mathbf{b}$ (parallel vectors).

> **Why it matters:** Bounds correlation $|r|\leq 1$, cosine similarity $\in[-1,1]$, and many
> optimisation convergence proofs.


In [ ]:
a = np.random.randn(5)
b = np.random.randn(5)
c = np.random.randn(1) * a   # c is parallel to a → equality case

a_dot_b = np.dot(a, b)
a_dot_c = np.dot(a, c)

lhs_ab = abs(a_dot_b)
rhs_ab = np.linalg.norm(a) * np.linalg.norm(b)

lhs_ac = abs(a_dot_c)
rhs_ac = np.linalg.norm(a) * np.linalg.norm(c)

print(f"|a·b| = {lhs_ab:.4f}  <=  ||a||·||b|| = {rhs_ab:.4f}  (strict inequality)")
print(f"|a·c| = {lhs_ac:.4f}  ~=  ||a||·||c|| = {rhs_ac:.4f}  (equality: c || a)")


## 11. Dot Product Sign and Scalar Multiplication

Scalars distribute into the dot product:

$$(\alpha\,\mathbf{a}) \cdot (\beta\,\mathbf{b}) = \alpha\beta\,(\mathbf{a}\cdot\mathbf{b})$$

Consequences:
- Negating either vector **flips the sign** of the dot product
- Scaling a vector **scales** the dot product by the same factor
- Orthogonal vectors **stay orthogonal** regardless of scaling ($0 \times k = 0$)


In [ ]:
v1 = np.array([-3,  4,  5], dtype=float)
v2 = np.array([ 3,  6, -3], dtype=float)
s1, s2 = -2.0, 3.0

original = np.dot(v1, v2)
scaled   = np.dot(s1 * v1, s2 * v2)
expected = s1 * s2 * original

print(f"v1 = {v1},  v2 = {v2}")
print(f"v1·v2                        = {original}")
print(f"(s1·v1)·(s2·v2)              = {scaled}")
print(f"s1·s2·(v1·v2) = {s1}×{s2}×{original} = {expected}")
print(f"Equal?                        {np.isclose(scaled, expected)}")


## 12. Hadamard Product (Element-wise Multiplication)

Multiplies element by element — result is a **vector**, not a scalar:

$$\mathbf{a} \odot \mathbf{b} =
\begin{bmatrix} a_1 b_1 \\ a_2 b_2 \\ \vdots \\ a_n b_n \end{bmatrix}$$

| Operation | Result | NumPy |
|-----------|--------|-------|
| Dot product $\mathbf{a}^\top\mathbf{b}$ | scalar | `a @ b` |
| Hadamard $\mathbf{a}\odot\mathbf{b}$ | vector $(n,)$ | `a * b` |

> **ML connections:** Attention masks, dropout (element-wise 0/1 mask), LSTM gating (sigmoid gates applied element-wise), gradient masking.


In [ ]:
a = np.array([1, 2, 3])
b = np.array([3, 4, 5])

hadamard = np.multiply(a, b)   # same as a * b
dot_prod  = a @ b

print(f"a               = {a}")
print(f"b               = {b}")
print(f"a ⊙ b (Hadamard) = {hadamard}  (vector, shape {hadamard.shape})")
print(f"a · b (dot)      = {dot_prod}       (scalar)")
print(f"a * b == np.multiply(a, b): {np.array_equal(a * b, hadamard)}")


## 13. The Outer Product

While the dot product *contracts* two vectors to a scalar, the outer product *expands* them to a matrix:

$$
\mathbf{v}\mathbf{w}^\top \in \mathbb{R}^{N \times M}, \qquad
(N\times 1)(1\times M) = (N\times M)
$$

The result is always **rank-1** (every row is a multiple of $\mathbf{w}^\top$).

> **ML connections:** Hebbian weight update $\Delta W = \eta\,\mathbf{x}\mathbf{y}^\top$;
> attention value accumulation; low-rank matrix factorisation (LoRA).


In [ ]:
v = np.array([1, 2, 3])
w = np.array([3, 4, 5, 6])

outer = np.outer(v, w)
print(f"v shape: {v.shape},  w shape: {w.shape}")
print(f"Outer product (shape {outer.shape}):\n{outer}")
print(f"Rank = {np.linalg.matrix_rank(outer)}  (always 1 for a pure outer product)")

v2 = np.array([3, 4, 5])
print(f"\nDot product v·v2 = {np.dot(v, v2)}  (scalar, requires same length)")


## 14. Cross Product (3-D Only)

Defined only for 3-D vectors; returns a vector **perpendicular to both** inputs:

$$
\mathbf{a} \times \mathbf{b} =
\begin{bmatrix} a_2 b_3 - a_3 b_2 \\ a_3 b_1 - a_1 b_3 \\ a_1 b_2 - a_2 b_1 \end{bmatrix}
$$

- **Magnitude:** $\|\mathbf{a}\times\mathbf{b}\| = \|\mathbf{a}\|\|\mathbf{b}\|\sin\theta$ — area of the parallelogram
- **Anti-commutative:** $\mathbf{a}\times\mathbf{b} = -(\mathbf{b}\times\mathbf{a})$
- **Zero iff parallel:** $\mathbf{a}\times\mathbf{b} = \mathbf{0} \Leftrightarrow \mathbf{a}\parallel\mathbf{b}$

> **Applications:** Surface normal vectors in 3-D graphics; torque in physics.


In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

cross_ab = np.cross(a, b)
cross_ba = np.cross(b, a)

print(f"a       = {a}")
print(f"b       = {b}")
print(f"a x b   = {cross_ab}")
print(f"b x a   = {cross_ba}  (= -(a x b))")

print(f"\nPerpendicularity check:")
print(f"  (a x b)·a = {np.dot(cross_ab, a)}  (must be 0)")
print(f"  (a x b)·b = {np.dot(cross_ab, b)}  (must be 0)")
print(f"\n||a x b|| = {np.linalg.norm(cross_ab):.4f}  (area of parallelogram)")

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')
O = [0, 0, 0]
for vec, col, lab in [(a, 'steelblue', 'a'), (b, 'tomato', 'b'), (cross_ab, 'forestgreen', 'axb')]:
    ax.quiver(*O, *vec, color=col, arrow_length_ratio=0.15, linewidth=2, label=lab)
ax.set(xlim=(-5, 5), ylim=(-5, 5), zlim=(-5, 5), title='Cross Product in 3-D')
ax.legend(); plt.tight_layout(); plt.show()


## 15. Complex Vectors

A **complex vector** $\mathbf{z} \in \mathbb{C}^n$ has complex-valued components.
Used in Fourier analysis, quantum mechanics, and signal processing.

**Conjugate** of $z = a + b\mathrm{i}$:
$$z^* = \bar{z} = a - b\mathrm{i}$$

**Hermitian transpose** (conjugate transpose, $\mathbf{z}^H$ or $\mathbf{z}^\dagger$):
$$\mathbf{z}^H = (\mathbf{z}^*)^\top \quad\text{— conjugate every element, then transpose}$$

**Why it matters for norms:**

$$\mathbf{z}^\top\mathbf{z} = (3+4\mathrm{i})^2 = -7+24\mathrm{i} \quad\text{(complex → not a valid norm)}$$
$$\mathbf{z}^H\mathbf{z} = |z|^2 = 25 \quad\text{(real and non-negative → correct squared norm)}$$

> **Note:** `np.matrix` is deprecated — always use `np.array` with `.conj().T`.


In [ ]:
# Scalar complex number
z_s = 3 + 4j
print(f"z           = {z_s}")
print(f"conjugate   = {z_s.conjugate()}")
print(f"|z|         = {abs(z_s)}")

# Complex vector using np.array (np.matrix is deprecated)
z = np.array([3+4j, 1-2j, 0+5j])
z_col = z.reshape(-1, 1)

print(f"\nz     = {z}")
print(f"z*    = {z.conj()}")

# Hermitian transpose = conjugate then transpose
z_H = z_col.conj().T
print(f"\nz column shape:        {z_col.shape}")
print(f"z^T (regular):         {z_col.T}")
print(f"z^H (Hermitian):       {z_H}")

# Compare z^T@z vs z^H@z for a scalar complex vector
s = np.array([[3+4j]])
via_T = (s.T @ s)[0, 0]
via_H = (s.conj().T @ s)[0, 0]
print(f"\nFor z = [3+4j]:")
print(f"  z^T @ z = {via_T}     <- complex, NOT a valid norm")
print(f"  z^H @ z = {via_H}   <- real, correct: ||z||^2 = {np.linalg.norm(s)**2:.0f}")


## 16. Dimensions, Fields, and Subspaces

**Dimensions** ($N$): the number of elements in $\mathbf{v} \in \mathbb{R}^N$.

**Field**: a set of numbers closed under $+$, $-$, $\times$, $\div$.
Common fields: $\mathbb{R}$ (ML), $\mathbb{C}$ (signal processing).

**Vector subspace** $V \subseteq \mathbb{R}^n$: closed under the parent-space operations.
$V$ is a subspace iff for all $\mathbf{u}, \mathbf{v} \in V$ and $\alpha, \beta \in \mathbb{R}$:

$$\alpha\mathbf{u} + \beta\mathbf{v} \in V$$

This implies $\mathbf{0} \in V$ (set $\alpha = \beta = 0$).

**Examples:**
- Any line through the origin in $\mathbb{R}^2$ ✅
- $\{(x, y) : x + y = 1\}$ ❌ (doesn't contain $\mathbf{0}$)


## 17. Span

The **span** of $\{\mathbf{v}_1, \ldots, \mathbf{v}_k\}$ is all their linear combinations:

$$\text{Span}(\{\mathbf{v}_1,\ldots,\mathbf{v}_k\})
= \left\{ \sum_{i=1}^k \alpha_i \mathbf{v}_i \;\middle|\; \alpha_i \in \mathbb{R} \right\}$$

**Geometric intuition:**

| Vectors | Span |
|---------|------|
| 1 vector | a **line** through the origin |
| 2 linearly independent vectors | a **plane** through the origin |
| 3 linearly independent vectors in $\mathbb{R}^3$ | all of $\mathbb{R}^3$ |


In [ ]:
v1 = np.array([1, 2, 0])
v2 = np.array([2, 1, 0])

alpha = np.linspace(-1.5, 1.5, 15)
beta  = np.linspace(-1.5, 1.5, 15)
alpha, beta = np.meshgrid(alpha, beta)

X = alpha * v1[0] + beta * v2[0]
Y = alpha * v1[1] + beta * v2[1]
Z = alpha * v1[2] + beta * v2[2]

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=Z, opacity=0.45, colorscale='Viridis', name='Span'))

for vec, col, name in [(v1, 'Reds', 'v1'), (v2, 'Blues', 'v2')]:
    fig.add_trace(go.Cone(x=[0], y=[0], z=[0],
                          u=[vec[0]], v=[vec[1]], w=[vec[2]],
                          colorscale=col, sizemode='absolute', sizeref=0.4,
                          name=name, showscale=False))

for a_val, b_val in [(-0.5, 1.0), (1.0, 0.5), (-1.0, -0.5)]:
    p = a_val * v1 + b_val * v2
    fig.add_trace(go.Scatter3d(x=[p[0]], y=[p[1]], z=[p[2]],
                               mode='markers', marker=dict(size=6, color='black'),
                               name=f'{a_val}*v1+{b_val}*v2'))

fig.update_layout(
    scene=dict(xaxis=dict(range=[-3, 3]),
               yaxis=dict(range=[-3, 3]),
               zaxis=dict(range=[-1, 1])),
    width=750, height=600,
    title='Span(v1, v2): a plane through the origin'
)
fig.show()


## 18. Linear Independence

$\{\mathbf{v}_1,\ldots,\mathbf{v}_n\}$ is **linearly independent** iff the only solution to

$$\lambda_1\mathbf{v}_1 + \cdots + \lambda_n\mathbf{v}_n = \mathbf{0}$$

is $\lambda_1 = \cdots = \lambda_n = 0$.

**How to test:**

| Method | Description |
|--------|-------------|
| Quick check | $n > \text{dim}$, zero vector, or duplicate → automatically dependent |
| **Matrix rank** | Form $A=[\mathbf{v}_1|\cdots|\mathbf{v}_n]$; independent iff $\text{rank}(A)=n$ |
| Coefficient solve | Solve $A\boldsymbol{\lambda}=\mathbf{0}$; independent iff trivial solution only |


In [ ]:
# Independent set
v1 = np.array([1, 2, 0])
v2 = np.array([3, 2, 1])
v3 = np.array([4, 6, 1])

A_indep = np.column_stack([v1, v2, v3])
r_i = np.linalg.matrix_rank(A_indep)
print(f"Independent set  -> rank = {r_i}/{A_indep.shape[1]} -> {'INDEPENDENT' if r_i == 3 else 'DEPENDENT'}")

# Dependent set: v3_dep = v1 + v2
v3_dep = v1 + v2
A_dep  = np.column_stack([v1, v2, v3_dep])
r_d    = np.linalg.matrix_rank(A_dep)
print(f"Dependent set    -> rank = {r_d}/{A_dep.shape[1]} -> {'INDEPENDENT' if r_d == 3 else 'DEPENDENT'}")

# Recover the dependency coefficients
coeffs, _, _, _ = np.linalg.lstsq(np.column_stack([v1, v2]), v3_dep, rcond=None)
print(f"\nv3_dep = {coeffs[0]:.2f}·v1 + {coeffs[1]:.2f}·v2")
print(f"Verify: {np.allclose(coeffs[0]*v1 + coeffs[1]*v2, v3_dep)}")


## Summary

| Concept | Formula | NumPy |
|---------|---------|-------|
| Vector length | $\|\mathbf{v}\|=\sqrt{\mathbf{v}^\top\mathbf{v}}$ | `np.linalg.norm(v)` |
| Dot product | $\mathbf{a}^\top\mathbf{b}=\sum a_i b_i$ | `a @ b` |
| Angle | $\cos\theta = \frac{\mathbf{a}^\top\mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$ | `np.arccos(a@b / (norm(a)*norm(b)))` |
| Unit vector | $\hat{\mathbf{v}}=\mathbf{v}/\|\mathbf{v}\|$ | `v / np.linalg.norm(v)` |
| Orthogonal | $\mathbf{a}\cdot\mathbf{b}=0$ | `np.isclose(a @ b, 0)` |
| Hadamard | $\mathbf{a}\odot\mathbf{b}$ (vector) | `a * b` |
| Outer product | $\mathbf{v}\mathbf{w}^\top$ (matrix) | `np.outer(v, w)` |
| Cross product | $\mathbf{a}\times\mathbf{b}$ (3-D only) | `np.cross(a, b)` |
| Hermitian transpose | $\mathbf{z}^H$ | `z.conj().T` |
| Linear independence | $\text{rank}(A)=n$ | `np.linalg.matrix_rank(A)` |

**Next →** Chapter 2: Matrices — rectangular arrays that represent linear transformations
and form the backbone of neural network weight matrices.
